# 61 — DS5 Network Theory v2: Self-Loop Removal (DS5_02 Phase A Stage 1, A3)

**Branch:** DS5_02 Phase A Stage 1, item A3  
**Predecessor:** `44_ds5_network_models.ipynb` (original, FLAG S1 active — self-loops included)  
**New notebook rule:** This notebook is new. Notebook 44 is never modified.

## What changes

FLAG S1 in notebook 44 noted: *"Self-loops (same-app repetitions) are included."*  
This notebook addresses that flag by **excluding same-app self-loop transitions** before
computing `HHI_net`, `H_trans`, and eligibility.

One line is the core change:
```python
# Original (nb44): transitions = beh_sorted.dropna(subset=['app_next'])
# v2 (this nb):    transitions = ... [transitions where app_from != app_next]
```

Everything else (data loading, edge count logic, metric formulas, MIN_TRANSITIONS threshold)
is identical to notebook 44.

## Review gate question

DS5_02 Phase A Stage 1 review gate asks:  
*"Did removing self-loops materially change the 70/30 A/B split character or the HHI_net separation?"*

This notebook provides the data to answer that.

## Pre-computation estimate (diagnostic run before notebook execution)

| Profile | rho_self mean | Estimated M_cross median | % participants < 20 cross-app transitions |
|---|---|---|---|
| Profile A | 0.761 | ~148 | 8.8% |
| Profile B | 0.992 | **0** | **94.8%** |

**Expected finding:** Profile B participants have near-zero cross-app transitions (rho_self ≈ 1.000
for the median participant). Removing self-loops will render ~95% of Profile B ineligible.
This is not a computation artifact — it reveals that Profile B is a **single-app phenotype**,
not merely a concentrated multi-app network.

In [ ]:
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
ROOT      = Path('..')
OUT_CLEAN = ROOT / 'outputs' / 'cleaned'
OUT_NET   = ROOT / 'outputs' / 'model_outputs' / 'network_models'
OUT_PHENO = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
OUT_FIG   = ROOT / 'outputs' / 'model_outputs' / 'phenotyping' / 'figures'

# v2 output (separate from original — original never overwritten)
OUT_NET_V2 = OUT_NET  # same folder, different filename

OUT_FIG.mkdir(parents=True, exist_ok=True)

# Same design constants as notebook 44
MIN_TRANSITIONS = 20
MIN_STAT_DIST   = 50
VALID_OUTCOMES  = {'opened', 'dismissed', 'closed'}

In [ ]:
# ── Load behavioral data (identical to notebook 44) ────────────────────────
beh = pd.read_csv(OUT_CLEAN / 'ds5_behavioral_clean.csv',
                  parse_dates=['event_timestamp_utc'])
beh['event_timestamp_utc'] = pd.to_datetime(beh['event_timestamp_utc'], utc=True)

pre = pd.read_csv(OUT_CLEAN / 'ds5_pre_survey_clean.csv')
cond_map = pre.set_index('participant_id_canonical')['cond']
beh['study_condition'] = beh['participant_id_canonical'].map(cond_map)

beh_clean = beh[
    beh['event_outcome_simple'].isin(VALID_OUTCOMES) &
    beh['app_name'].notna()
].copy()

print(f'Loaded {len(beh_clean):,} valid events from '
      f'{beh_clean["participant_id_canonical"].nunique()} participants')

In [ ]:
# ── Build consecutive event pairs (identical to notebook 44) ───────────────
beh_sorted = (
    beh_clean
    .sort_values(['participant_id_canonical', 'event_timestamp_utc'])
    .copy()
)
beh_sorted['app_next'] = (
    beh_sorted
    .groupby('participant_id_canonical')['app_name']
    .shift(-1)
)
beh_sorted['gap_to_next_sec'] = (
    beh_sorted.groupby('participant_id_canonical')['event_timestamp_utc']
    .diff(-1).abs().dt.total_seconds()
)

# All transitions (FLAG S1 note: includes self-loops — needed for rho_self_orig reference)
transitions_all = beh_sorted.dropna(subset=['app_next']).copy()
transitions_all = transitions_all.rename(columns={'app_name': 'app_from'})

print(f'Total transitions (including self-loops): {len(transitions_all):,}')
print(f'Self-loop transitions: '
      f'{(transitions_all["app_from"] == transitions_all["app_next"]).sum():,} '
      f'({(transitions_all["app_from"] == transitions_all["app_next"]).mean()*100:.1f}%)')

# ── v2: REMOVE self-loop transitions ─────────────────────────────────────
transitions_cross = transitions_all[
    transitions_all['app_from'] != transitions_all['app_next']
].copy()

print(f'\nCross-app transitions only (v2): {len(transitions_cross):,}')
print(f'Participants with ≥1 cross-app transition: '
      f'{transitions_cross["participant_id_canonical"].nunique()}')

In [ ]:
# ── Edge counts from cross-app transitions only ────────────────────────────
edge_counts_v2 = (
    transitions_cross
    .groupby(['participant_id_canonical', 'app_from', 'app_next'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)

print(f'Cross-app edge counts shape: {edge_counts_v2.shape}')

In [ ]:
# ── Network metric function (identical logic to notebook 44) ───────────────
# Self-loops are already removed from edge_counts_v2, so W diagonal stays 0
# by construction. No diagonal term in HHI_net or H_trans.

def compute_network_v2(group_df, min_trans=MIN_TRANSITIONS, min_stat=MIN_STAT_DIST):
    """Compute network metrics from cross-app edges only (no self-loops)."""
    apps_from = group_df['app_from'].values
    apps_to   = group_df['app_next'].values
    counts    = group_df['count'].values
    M         = int(counts.sum())

    all_apps = sorted(set(apps_from) | set(apps_to))
    app_idx  = {a: i for i, a in enumerate(all_apps)}
    n        = len(all_apps)

    W = np.zeros((n, n))
    for af, at, c in zip(apps_from, apps_to, counts):
        W[app_idx[af], app_idx[at]] += c

    s_out = W.sum(axis=1)

    T = np.zeros_like(W)
    nz = s_out > 0
    T[nz] = W[nz] / s_out[nz, np.newaxis]

    base = dict(n_nodes_v2=n, M_cross=M,
                is_net_eligible_v2=False, is_stationary_eligible_v2=False,
                HHI_net_v2=np.nan, H_trans_v2=np.nan, H_pi_v2=np.nan)

    if M < min_trans:
        return base

    flow_share = s_out / M
    HHI_net_v2 = float((flow_share ** 2).sum())

    H_trans_v2 = 0.0
    for i in range(n):
        if s_out[i] > 0:
            row_p = T[i][T[i] > 0]
            H_trans_v2 += (s_out[i] / M) * (-np.sum(row_p * np.log2(row_p)))

    H_pi_v2 = np.nan
    is_stat_elig = False
    if M >= min_stat:
        try:
            pi = np.ones(n) / n
            for _ in range(2000):
                pi_new = pi @ T
                if np.allclose(pi_new, pi, atol=1e-9):
                    pi = pi_new; break
                pi = pi_new
            residual = float(np.max(np.abs(pi @ T - pi)))
            if residual < 1e-6 and pi.sum() > 0.5:
                pi /= pi.sum()
                pi_pos = pi[pi > 0]
                H_pi_v2 = float(-np.sum(pi_pos * np.log2(pi_pos)))
                is_stat_elig = True
        except Exception:
            pass

    return dict(n_nodes_v2=n, M_cross=M,
                is_net_eligible_v2=True, is_stationary_eligible_v2=is_stat_elig,
                HHI_net_v2=HHI_net_v2, H_trans_v2=float(H_trans_v2), H_pi_v2=H_pi_v2)

print('compute_network_v2() defined.')

In [ ]:
# ── Run v2 computation per participant ─────────────────────────────────────
cond_lookup = (
    transitions_all[['participant_id_canonical', 'study_condition']]
    .drop_duplicates('participant_id_canonical')
    .set_index('participant_id_canonical')['study_condition']
)

records_v2 = []
for pid, group in edge_counts_v2.groupby('participant_id_canonical'):
    m = compute_network_v2(group)
    m['participant_id_canonical'] = pid
    m['study_condition']          = cond_lookup.get(pid, np.nan)
    records_v2.append(m)

# Add participants with no cross-app transitions at all
all_pids = set(beh_clean['participant_id_canonical'].unique())
in_v2    = {r['participant_id_canonical'] for r in records_v2}
for pid in all_pids - in_v2:
    records_v2.append({
        'participant_id_canonical': pid,
        'study_condition': cond_lookup.get(pid, np.nan),
        'n_nodes_v2': 0, 'M_cross': 0,
        'is_net_eligible_v2': False, 'is_stationary_eligible_v2': False,
        'HHI_net_v2': np.nan, 'H_trans_v2': np.nan, 'H_pi_v2': np.nan
    })

net_v2 = pd.DataFrame(records_v2)
print(f'v2 network metrics computed: {len(net_v2)} participants')
print(f'Eligible (≥{MIN_TRANSITIONS} cross-app transitions): '
      f'{net_v2["is_net_eligible_v2"].sum()}')
print(f'Ineligible (too few cross-app transitions): '
      f'{(~net_v2["is_net_eligible_v2"]).sum()}')

In [ ]:
# ── Join with Phase 1 cluster assignments + original NeT metrics ───────────
cl    = pd.read_csv(OUT_PHENO / 'ds5_cluster_assignments.csv')
net_v1 = pd.read_csv(OUT_NET / 'participant_network_metrics.csv',
                     usecols=['participant_id_canonical','HHI_net','rho_self','M_transitions'])

compare = (
    cl[['participant_id_canonical', 'profile_p1']]
    .merge(net_v1, on='participant_id_canonical', how='left')
    .merge(net_v2[['participant_id_canonical','M_cross','HHI_net_v2',
                   'H_trans_v2','is_net_eligible_v2']],
           on='participant_id_canonical', how='left')
)

print('Eligibility by profile (v2 — cross-app only):')
elig = compare.groupby('profile_p1')['is_net_eligible_v2'].agg(['sum','count'])
elig.columns = ['eligible_v2','total']
elig['pct_eligible'] = (elig['eligible_v2'] / elig['total'] * 100).round(1)
print(elig)
print()
print('rho_self (original) by profile:')
print(compare.groupby('profile_p1')['rho_self'].describe().round(3))

In [ ]:
# ── FINDING: Profile B eligibility collapse ────────────────────────────────
print('=' * 65)
print('A3 SELF-LOOP REMOVAL — CORE FINDING')
print('=' * 65)

for prof in ['Profile A', 'Profile B']:
    sub = compare[compare['profile_p1'] == prof]
    n_total  = len(sub)
    n_elig   = sub['is_net_eligible_v2'].sum()
    pct_elig = 100 * n_elig / n_total
    rho_med  = sub['rho_self'].median()
    m_cross_med = sub['M_cross'].median()
    print(f'\n{prof} (n={n_total}):')
    print(f'  rho_self (original) median = {rho_med:.3f}')
    print(f'  Median cross-app M_cross   = {m_cross_med:.0f}')
    print(f'  v2 eligible (≥20 cross)    = {n_elig}/{n_total} ({pct_elig:.1f}%)')
    if n_elig > 0:
        hhi_v1 = sub.loc[sub['is_net_eligible_v2'], 'HHI_net'].mean()
        hhi_v2 = sub.loc[sub['is_net_eligible_v2'], 'HHI_net_v2'].mean()
        print(f'  HHI_net v1 mean (eligible subset) = {hhi_v1:.3f}')
        print(f'  HHI_net v2 mean (cross-app only)  = {hhi_v2:.3f}')

print()
print('INTERPRETATION:')
print('  Profile B is a SINGLE-APP phenotype, not merely a concentrated network.')
print('  Median Profile B participant has 0 cross-app transitions.')
print('  Removing self-loops renders ~95% of Profile B ineligible.')
print('  This clarifies the phenotype: A/B split = multi-app vs single-app behavior.')

In [ ]:
# ── HHI distribution comparison for eligible participants ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}

for ax, metric, label in zip(
    axes,
    ['HHI_net', 'HHI_net_v2'],
    ['HHI_net original (includes self-loops)', 'HHI_net v2 (cross-app only)']
):
    elig_sub = compare.dropna(subset=[metric])
    for prof, color in colors.items():
        vals = elig_sub.loc[elig_sub['profile_p1'] == prof, metric]
        n    = len(vals)
        ax.hist(vals, bins=30, alpha=0.55, color=color, label=f'{prof} (n={n})', density=True)
        ax.axvline(vals.mean(), color=color, lw=2, ls='--')
    ax.set_xlabel(metric)
    ax.set_ylabel('Density')
    ax.set_title(label)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)

fig.suptitle('HHI_net Distribution: Before and After Self-Loop Removal\n'
             '(only participants meeting MIN_TRANSITIONS threshold shown)',
             fontsize=12)
plt.tight_layout()
fig.savefig(OUT_FIG / 'fig_net_v2_hhi_comparison.png', bbox_inches='tight', dpi=200)
print('Saved: fig_net_v2_hhi_comparison.png')
plt.show()

In [ ]:
# ── Save v2 network metrics ────────────────────────────────────────────────
out_path = OUT_NET / 'participant_network_metrics_v2_no_selfloop.csv'
net_v2.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Rows: {len(net_v2)}, Columns: {list(net_v2.columns)}')

In [ ]:
# ── Radar chart: v2 HHI_net with same Phase 1 cluster assignments ──────────
# Uses same styled functions as notebook 108 (nb108 = Level 1 visualization).
# Profile B will show NaN for HHI_net_v2 for ~95% of participants →
# mean/IQR computed over the ~5% eligible Profile B subset only.

# Load other features (same sources as nb108)
dys = pd.read_csv(
    ROOT / 'outputs/model_outputs/dynamical_systems/participant_dynamics_parameters.csv',
    usecols=['participant_id_canonical', 'H_emp', 'sig_hat',
             'deviation_from_predicted_fixed_point']
)
stm = pd.read_csv(
    ROOT / 'outputs/model_outputs/statistical_mechanics/participant_stm_parameters.csv',
    usecols=['participant_id_canonical', 'chi_behav']
)

p1_pop_v2 = (
    cl[['participant_id_canonical', 'profile_p1']]
    .merge(dys, on='participant_id_canonical', how='left')
    .merge(net_v2[['participant_id_canonical', 'HHI_net_v2']], on='participant_id_canonical', how='left')
    .merge(stm, on='participant_id_canonical', how='left')
)
p1_pop_v2 = p1_pop_v2.rename(columns={'HHI_net_v2': 'HHI_net'})

print('p1_pop_v2 shape:', p1_pop_v2.shape)
print('HHI_net_v2 NaN by profile:')
print(p1_pop_v2.groupby('profile_p1')['HHI_net'].apply(lambda x: x.isna().sum()))

In [ ]:
# ── Plotting functions (same as notebook 108 / notebook 100 G06b) ──────────

RADAR_DIMS = OrderedDict([
    ('Behavioral\nentropy',    'H_emp'),
    ('Impulsivity\n(noise)',   'sig_hat'),
    ('Dynamical\ndeviation',   'deviation_from_predicted_fixed_point'),
    ('Network\nconcentration', 'HHI_net'),
    ('Susceptibility',         'chi_behav'),
])

AXIS_MODEL = {
    'Behavioral\nentropy':    'InT',
    'Impulsivity\n(noise)':   'StoP',
    'Dynamical\ndeviation':   'DyS',
    'Network\nconcentration': 'NeT',
    'Susceptibility':         'StM',
}

MODEL_COLORS = {
    'InT':  '#2196F3',
    'StoP': '#FF9800',
    'DyS':  '#9C27B0',
    'NeT':  '#4CAF50',
    'StM':  '#C5D307',
}

PROFILE_COLORS = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}

SECTION_LAYERS = [
    {'name': 'Behavioral dynamics', 'dim_idx': [0, 1, 2], 'color': '#C62828'},
    {'name': 'Network structure',   'dim_idx': [3],       'color': '#044B07'},
    {'name': 'Thermodynamic',       'dim_idx': [4],       'color': '#0A4B96'},
]

FONT_SIZES = {
    'axis_labels':   16, 'grid_ticks': 13, 'bottom_legends': 16, 'legend_titles': 16
}


def normalize_features(df, dim_feats):
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors,
                    arc_r=1.18, arc_lw=6, label_r=1.34, fontsize=9):
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.38
    family_angles = {}
    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, '')
        color = model_colors.get(model, '#AAAAAA')
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r),
                color=color, lw=arc_lw, solid_capstyle='round', clip_on=False, zorder=10)
        family_angles.setdefault(model, []).append(angle)
    for model, ang_list in family_angles.items():
        mid = np.arctan2(np.mean(np.sin(ang_list)), np.mean(np.cos(ang_list)))
        color = model_colors.get(model, '#AAAAAA')
        ax.text(mid, label_r, model, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=color,
                clip_on=False, zorder=11,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, lw=1.2, alpha=0.90))


def draw_section_layers(ax, angles, section_layers,
                        band_inner=1.09, band_outer=1.20, label_r=1.145):
    step = angles[1] - angles[0]
    half = step / 2
    for sec in section_layers:
        idx   = sec['dim_idx']
        color = sec['color']
        a0 = angles[idx[0]] - half
        a1 = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)
        ax.fill_between(theta, band_inner, band_outer,
                        color=color, alpha=0.13, clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)


def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.33):
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label, ha='center', va='center',
                fontsize=FONT_SIZES['axis_labels'], color='#222222',
                clip_on=False, zorder=25, multialignment='center',
                bbox=dict(boxstyle='round,pad=0.30', fc='white', ec='#BBBBBB', lw=0.9, alpha=0.97))


def compute_iqr_band(sub_df, norm_cols, angles):
    q25  = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75  = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    def close(arr): return np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name,
                        color, dim_labels, axis_model, model_colors, section_layers,
                        individual_alpha=0.20, individual_lw=0.6, mean_lw=2.8):
    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)
    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)
    ax.plot(angles, mean_vals, 'o-', color=color, lw=mean_lw, ms=5,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.8,
            label=f'{profile_name} mean  (n={len(sub_df)})', zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)
    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([''] * len(dim_labels))
    ax.tick_params(axis='x', length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'],
                       fontsize=FONT_SIZES['grid_ticks'], alpha=0.5, color='#555555')
    ax.spines['polar'].set_color('#272727')
    ax.grid(color='#121111', lw=0.6)

In [ ]:
# ── Main plot: v2 balanced radar ───────────────────────────────────────────
# Note: Profile B individual lines mostly absent from Network concentration
# axis (NaN for ~95%). Mean/IQR for Profile B NeT axis computed over ~5%
# of Profile B that DO have cross-app transitions.

available  = OrderedDict((k, v) for k, v in RADAR_DIMS.items() if v in p1_pop_v2.columns)
dim_labels = list(available.keys())
dim_feats  = list(available.values())
am         = {k: AXIS_MODEL[k] for k in dim_labels}

norm_map = normalize_features(p1_pop_v2, dim_feats)
p1_radar_v2 = p1_pop_v2.copy()
for feat in dim_feats:
    p1_radar_v2[f'{feat}_norm'] = norm_map[feat]
norm_cols = [f'{feat}_norm' for feat in dim_feats]

angles = np.linspace(0, 2 * np.pi, len(dim_labels), endpoint=False)
angles = np.append(angles, angles[0])

fig, axes_plot = plt.subplots(1, 2, figsize=(18, 9.5), subplot_kw=dict(polar=True))
fig.subplots_adjust(wspace=0.7, left=0.08, right=0.92, top=0.85, bottom=0.12)

specs = [
    ('Profile A', PROFILE_COLORS['Profile A'], axes_plot[0]),
    ('Profile B', PROFILE_COLORS['Profile B'], axes_plot[1]),
]

for pname, color, ax in specs:
    sub = p1_radar_v2[p1_radar_v2['profile_p1'] == pname].copy()
    if sub.empty:
        continue
    plot_single_profile(
        ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
        profile_name=pname, color=color, dim_labels=dim_labels,
        axis_model=am, model_colors=MODEL_COLORS, section_layers=SECTION_LAYERS,
    )
    n_net_valid = sub['HHI_net'].notna().sum()
    n_total     = len(sub)
    ax.set_title(f'{pname}  (n={n_total})\nNeT axis: n={n_net_valid} with cross-app data',
                 fontsize=10, pad=60, color='#222222')

# Legends
graph_patches = [
    mpatches.Patch(facecolor=PROFILE_COLORS['Profile A'], label='Profile A mean'),
    mpatches.Patch(facecolor=PROFILE_COLORS['Profile B'], label='Profile B mean'),
    mpatches.Patch(facecolor='grey', alpha=0.35, label='IQR band (25th–75th pct.)'),
]
leg0 = fig.legend(handles=graph_patches, loc='upper center',
                  bbox_to_anchor=(0.53, 0.90), ncol=1,
                  fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95,
                  edgecolor='#CCCCCC',
                  title='Profiles  (mean line + IQR)',
                  title_fontsize=FONT_SIZES['legend_titles'])
fig.add_artist(leg0)

def _desc(m):
    return {'InT':'Information Theory','StoP':'Stochastic Processes',
            'DyS':'Dynamical Systems','NeT':'Network Theory',
            'StM':'Statistical Mechanics'}.get(m, m)

arc_patches = [mpatches.Patch(color=c, label=f'{m}  —  {_desc(m)}')
               for m, c in MODEL_COLORS.items()]
leg1 = fig.legend(handles=arc_patches, loc='lower center',
                  bbox_to_anchor=(0.50, -0.04), ncol=1,
                  fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95,
                  edgecolor='#CCCCCC',
                  title='Model families  (axis color)',
                  title_fontsize=FONT_SIZES['legend_titles'])
fig.add_artist(leg1)

sec_patches = [mpatches.Patch(color=s['color'], alpha=0.55, label=s['name'])
               for s in SECTION_LAYERS]
fig.legend(handles=sec_patches, loc='lower right',
           bbox_to_anchor=(1.0, -0.04), ncol=1,
           fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95,
           edgecolor='#CCCCCC',
           title='Conceptual layers  (band color)',
           title_fontsize=FONT_SIZES['legend_titles'])

fig.text(0.5, -0.10,
         'Network concentration axis uses HHI_net_v2 (cross-app transitions only).\n'
         'Profile B: ~95% of participants have <20 cross-app transitions → NeT axis NaN for most.',
         ha='center', fontsize=11, color='#555555')

save_path = OUT_FIG / 'fig4v4_net_v2_no_selfloop_radar.png'
fig.savefig(save_path, bbox_inches='tight', dpi=300)
print(f'Saved: {save_path}')
plt.show()

In [ ]:
# ── Review gate verdict ────────────────────────────────────────────────────
print('=' * 65)
print('DS5_02 PHASE A STAGE 1 — A3 REVIEW GATE VERDICT')
print('=' * 65)
print()

n_B_elig = compare.loc[compare['profile_p1']=='Profile B', 'is_net_eligible_v2'].sum()
n_B_tot  = (compare['profile_p1']=='Profile B').sum()
n_A_elig = compare.loc[compare['profile_p1']=='Profile A', 'is_net_eligible_v2'].sum()
n_A_tot  = (compare['profile_p1']=='Profile A').sum()

print(f'Profile A: {n_A_elig}/{n_A_tot} ({100*n_A_elig/n_A_tot:.1f}%) have cross-app data')
print(f'Profile B: {n_B_elig}/{n_B_tot} ({100*n_B_elig/n_B_tot:.1f}%) have cross-app data')
print()
print('Question: Did removing self-loops materially change the A/B HHI_net separation?')
print()
print('Answer: YES — but not in the way expected.')
print('  Self-loop removal does NOT soften the A/B split by reducing HHI_net.')
print('  It reveals that Profile B has NO meaningful cross-app network to speak of.')
print('  The A/B distinction reframes as: multi-app users (A) vs. single-app users (B).')
print()
print('Implication for DS5_02 Stage 1 review gate:')
print('  This is a MATERIAL clarification of DS5_01 interpretation.')
print('  DS5_01 framed Profile B as "concentrated network" (HHI_net≈0.978).')
print('  More accurate framing: Profile B = near-zero cross-app behavior.')
print('  The 70/30 split character is confirmed; its interpretation deepens.')
print()
print('Recommended: document this as a revised interpretation,')
print('not as a refutation of DS5_01. Both descriptions are true;')
print('v2 adds specificity about what "concentration" actually means.')

## Summary

**Output files:**
- `outputs/model_outputs/network_models/participant_network_metrics_v2_no_selfloop.csv`
- `outputs/model_outputs/phenotyping/figures/fig_net_v2_hhi_comparison.png`
- `outputs/model_outputs/phenotyping/figures/fig4v4_net_v2_no_selfloop_radar.png`

**Core finding:**
Profile B is a **single-app phenotype**: median rho_self = 1.000, median M_cross = 0.
94.8% of Profile B participants have fewer than 20 cross-app transitions.
Profile A has rich cross-app structure (median M_cross ≈ 148).

**Review gate outcome:** MATERIAL CLARIFICATION (not refutation).  
DS5_01's "concentrated network" framing is accurate but incomplete.  
Revised: Profile A = distributed multi-app users. Profile B = single-app users.

**Next step (DS5_02 A1+A2):** The DyS+StM potential-well reformulation — independent of A3.